![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 3. Parte B: Transferencia de estilos rápida

**Asignatura:** Redes Neuronales Profundas
**Bloque:** 3 — Transferencia de conocimiento

---

## Introducción

En la clase vimos que una red convolucional preentrenada no solo sirve para clasificar: sus capas intermedias son **extractores de features** reutilizables para tareas muy distintas a la original. Una de las aplicaciones más vistosas de esta idea es la **transferencia de estilos**: combinar el *contenido* de una foto con el *estilo* de una pintura en una sola imagen sintetizada.

Hay dos familias de métodos para hacerlo:

- **Gatys et al. 2015** — optimizar directamente los píxeles de una imagen sintetizada, usando una VGG preentrenada para medir la distancia de contenido y de estilo. Es el método que estudiamos en la teoría. Es flexible (cualquier par contenido/estilo) pero **lento**: cada imagen nueva requiere muchas de iteraciones de descenso de gradiente.
- **Johnson et al. 2016** — entrenar **una red feed-forward** (TransformerNet) especializada en un único estilo. Una vez entrenada, estilizar una imagen nueva es un solo forward pass: *tiempo real*. El precio: una red entrenada por estilo.

En este laboratorio vas a implementar el método de Johnson. Vas a:

- Construir las piezas que ambos métodos comparten: matriz de Gram, pérdida de contenido y pérdida de estilo sobre features de VGG.
- Ensamblar una **TransformerNet** encoder-decoder con bloques residuales.
- Entrenarla sobre COCO para que aprenda a reproducir el estilo de una única imagen.
- Usarla para estilizar imágenes nuevas y analizar el resultado.

> **Importante — GPU:** este laboratorio entrena una red convolucional sobre ~5.000 imágenes. **Activá la GPU en Colab** antes de empezar: *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*. Sin GPU el entrenamiento es impracticable (horas en lugar de minutos).

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de la clase (en particular la sección de transferencia de estilos).

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas (enunciados, explicaciones, ejemplos provistos y el encabezado) **no se tocan**: la corrección se hace celda por celda de manera automática y modificar lo que no corresponde puede invalidar tu entrega.

Si necesitás probar algo fuera de una celda de actividad, hacelo en una copia aparte y revertí los cambios antes de entregar.

In [ ]:
# ─── Setup: imports y detección de GPU ──────────────────────────────────────
# Estos son los imports que usa el laboratorio de punta a punta:
#   - torch / torchvision: red preentrenada (VGG-16), capas, optimizadores y
#     transformaciones de imágenes.
#   - cv2 / PIL / matplotlib: cargar, convertir y mostrar imágenes.
#   - os / urllib: descargar dataset (COCO val2017) e imagen de estilo.
# También detectamos si hay GPU disponible y la guardamos en `device`: esa
# variable la usás en todos los `.to(device)` del laboratorio.
import os
import random
import time
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms

device = (
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()        else
    "cpu"
)
print(f"Versión de PyTorch: {torch.__version__}")
print(f"Dispositivo:        {device}")
if device == "cpu":
    print("ADVERTENCIA: sin GPU el entrenamiento va a ser inviable. "
          "Activá la GPU en Colab (T4) o corré local con CUDA/MPS antes de continuar.")

In [ ]:
# ─── Setup: descarga del dataset de contenido (COCO val2017) ────────────────
# Usamos el split val2017 (~5.000 imágenes, ~780 MB) en lugar de train2014
# (13 GB) del paper original: alcanza para que la red aprenda un estilo
# razonable en una sola pasada de ~5 minutos sobre T4.
os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/val2017'):
    !wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
    !unzip -q /content/val2017.zip -d /content/dataset/
    !rm /content/val2017.zip
else:
    print('Dataset ya descargado')
print(f'Imágenes disponibles: {len(os.listdir("/content/dataset/val2017"))}')

In [ ]:
# ─── Setup: imagen de estilo ────────────────────────────────────────────────
# Por defecto bajamos una imagen de estilo clásica (mosaic.jpg) desde el repo
# oficial pytorch/examples. Podés cambiar STYLE_NAME a 'candy', 'rain_princess'
# o 'udnie' para probar otros estilos.
STYLES = {
    'mosaic':        'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/mosaic.jpg',
    'candy':         'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/candy.jpg',
    'rain_princess': 'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/rain-princess.jpg',
    'udnie':         'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/udnie.jpg',
}

STYLE_NAME = 'mosaic'
style_path = f'/content/style_images/{STYLE_NAME}.jpg'
os.makedirs('/content/style_images', exist_ok=True)

if not os.path.exists(style_path):
    req = urllib.request.Request(STYLES[STYLE_NAME],
                                 headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=20) as r, open(style_path, 'wb') as f:
        f.write(r.read())

plt.figure(figsize=(8, 5))
plt.imshow(Image.open(style_path))
plt.title(f'Imagen de estilo: {STYLE_NAME}')
plt.axis('off')
plt.show()

In [ ]:
# ─── Setup: helpers de imagen ───────────────────────────────────────────────
# Funciones auxiliares para leer imágenes en BGR (formato de cv2, coincide
# con el preprocesamiento Caffe que espera la VGG que usamos más abajo),
# convertirlas a tensores en rango [0, 255] con dimensión de batch, y volver.
def load_image(path):
    """Devuelve una imagen BGR (numpy, shape H, W, 3)."""
    return cv2.imread(path)


def itot(img, max_size=None):
    """Convierte una imagen numpy BGR en un tensor (1, 3, H, W) en [0, 255]."""
    if max_size is None:
        t = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.mul(255)),
        ])
    else:
        H, W, _ = img.shape
        size = tuple(int((float(max_size) / max(H, W)) * x) for x in [H, W])
        t = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(size),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.mul(255)),
        ])
    return t(img).unsqueeze(0)


def ttoi(tensor):
    """Convierte un tensor (1, 3, H, W) en un array numpy (H, W, 3)."""
    return tensor.squeeze().cpu().numpy().transpose(1, 2, 0)

---
## Sección A: Fundamentos compartidos (Gatys y Johnson)

### Una misma idea para dos métodos

Tanto Gatys como Johnson usan una VGG preentrenada para calcular dos pérdidas:

- **Pérdida de contenido**: MSE entre los *feature maps* de una capa intermedia de la imagen generada y de la imagen de contenido. Mide cuánto se parecen **qué cosas hay** y **dónde están**.
- **Pérdida de estilo**: MSE entre las **matrices de Gram** de los feature maps en varias capas. La matriz de Gram captura **correlaciones entre canales** e ignora la posición espacial — justamente lo que asociamos con "estilo".

En esta sección vas a implementar esas dos piezas y después las vas a reutilizar en el loop de entrenamiento del método de Johnson. Son las mismas funciones que usarías en el método de Gatys.

### Ejercicio 1 — Matriz de Gram

**Objetivo:** Implementar la operación central para medir estilo: la matriz de Gram de un tensor de feature maps.

**Contexto:**

Dado un tensor de activaciones de forma `(B, C, H, W)` (batch, canales, alto, ancho), la matriz de Gram se obtiene aplanando el eje espacial y haciendo el producto externo entre canales:

$$G = \frac{X \, X^\top}{C \cdot H \cdot W}$$

donde $X$ es el tensor reshapeado a `(B, C, H*W)`. El resultado tiene forma `(B, C, C)`: el elemento `G[b, i, j]` es la correlación entre los canales $i$ y $j$ en el ejemplo $b$. Dividimos por `C*H*W` para que la escala no dependa del tamaño de entrada.

**Enunciado:**

1. Implementá la función `gram(tensor)` que reciba un tensor de forma `(B, C, H, W)` y devuelva su matriz de Gram normalizada de forma `(B, C, C)`.
2. Para el producto `X · X^T` por ejemplo del batch, usá `torch.bmm` (*batched matrix multiplication*).
3. Probá la función con un tensor aleatorio de forma `(2, 4, 8, 8)` e imprimí:
   - La forma del resultado.
   - Si la matriz resultante es simétrica (una matriz de Gram siempre lo es).

> **Pista:** `tensor.view(B, C, H*W)` te aplana el eje espacial sin copiar memoria. Para chequear simetría: `torch.allclose(G, G.transpose(1, 2))`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

La matriz de Gram se construye justamente colapsando la dimensión espacial (`H, W`) y quedándose con la correlación entre canales. ¿Por qué eso captura "estilo" y descarta "contenido"? Pensalo imaginando qué pasa si tomás una imagen y la desplazás o la rotás un poco: ¿cambia mucho su matriz de Gram? ¿Cambian mucho sus feature maps directos?

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Pérdidas de contenido y estilo sobre features de VGG

**Objetivo:** Definir las dos pérdidas perceptuales que usan Gatys y Johnson, operando sobre **dicts de features** extraídos de distintas capas de VGG.

**Contexto:**

Cuando calculamos estas pérdidas, tenemos dict de features `{'relu1_2': ..., 'relu2_2': ..., 'relu3_3': ..., 'relu4_3': ...}` para la imagen generada y para la(s) imagen(es) de referencia. El convenio del paper original es:

- **Contenido**: MSE sobre `relu2_2` (una sola capa, intermedia).
- **Estilo**: suma de MSE entre matrices de Gram sobre las **cuatro** capas.

**Enunciado:**

Implementá dos funciones. Asumí que `gram` ya está disponible (del ejercicio 1) y que `mse = nn.MSELoss()` ya está instanciado al principio de la celda.

1. `content_loss(feats_gen, feats_content)`: MSE entre `feats_gen['relu2_2']` y `feats_content['relu2_2']`.
2. `style_loss(feats_gen, style_gram)`: donde `style_gram` es un dict con las matrices de Gram **ya precomputadas** de la imagen de estilo (una por capa). Para cada clave `k` en `feats_gen`, calculá la Gram del generado y acumulá su MSE contra `style_gram[k]`. Devolvé la suma.

Probalas con tensores dummy de la forma correcta y verificá que devuelven escalares no negativos.

> **Pistas:**
> - Precomputar `style_gram` es una optimización clave: la imagen de estilo es fija durante todo el entrenamiento, así que no tiene sentido recalcular su Gram en cada iteración.
> - Para el test dummy podés generar tensores de shape `(4, 64, 32, 32)` y similares — la forma exacta no importa, solo que tenga 4 ejes.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Una alternativa ingenua para medir cuánto se parecen dos imágenes es calcular el MSE **píxel a píxel** entre ellas. Sin embargo, ni Gatys ni Johnson usan eso: ambos miden pérdidas sobre **features de VGG** (una red preentrenada para clasificar ImageNet). ¿Por qué? Pensalo en términos de qué querés que la pérdida "considere cercano" y qué no.

*(Escribí tu respuesta acá)*

---
## Sección B: TransformerNet — la red que transforma imágenes

### Una red feed-forward para estilizar

En el método de Gatys, cada imagen nueva requiere un proceso de optimización por descenso de gradiente sobre sus píxeles (cientos de iteraciones). Johnson propone reemplazar ese proceso por una **única red** que aprenda a hacer la transformación en un solo forward pass.

La arquitectura elegida es una CNN encoder-decoder con **bloques residuales** en el medio:

- **Encoder**: 3 convoluciones que reducen la resolución 4× y suben los canales (3 → 32 → 64 → 128).
- **5 bloques residuales** de 128 canales: aprenden *correcciones estilísticas* sin destruir la estructura del contenido (los skip connections garantizan que la identidad sea una solución trivial).
- **Decoder**: 2 deconvoluciones (upsampling) + 1 convolución final que restauran la resolución original y vuelven a 3 canales.

> **Detalle histórico:** la clase se llama `TransformerNetwork`, pero **no** es un Transformer (attention). Es una CNN que "transforma" una imagen. El nombre es previo al paper de Vaswani (2017).

In [ ]:
# ─── Setup: bloques de construcción de la TransformerNet ───────────────────
# Las tres clases de abajo son piezas estándar y no es parte del objetivo
# pedagógico que las escribas. Leé los docstrings para saber qué hace cada una
# — las vas a usar en el ejercicio 3 para ensamblar la red completa.
class ConvLayer(nn.Module):
    """
    Convolución con reflection padding + InstanceNorm (o BatchNorm, o sin norm).
    El reflection padding evita los bordes negros que genera el padding cero
    convencional; la instance norm estabiliza el entrenamiento de redes de
    generación de imágenes (Ulyanov et al. 2016).
    """
    def __init__(self, in_c, out_c, kernel, stride, norm='instance'):
        super().__init__()
        self.reflection_pad = nn.ReflectionPad2d(kernel // 2)
        self.conv = nn.Conv2d(in_c, out_c, kernel, stride)
        self.norm_type = norm
        if norm == 'instance':
            self.norm = nn.InstanceNorm2d(out_c, affine=True)
        elif norm == 'batch':
            self.norm = nn.BatchNorm2d(out_c, affine=True)

    def forward(self, x):
        x = self.conv(self.reflection_pad(x))
        return x if self.norm_type == 'None' else self.norm(x)


class ResidualLayer(nn.Module):
    """
    Bloque residual (He et al. 2015): dos ConvLayer 3x3 con ReLU entre medio y
    un skip connection aditivo. Permite apilar muchos bloques sin degradar el
    entrenamiento (el gradiente tiene un "atajo" a través del skip).
    """
    def __init__(self, channels=128, kernel=3):
        super().__init__()
        self.conv1 = ConvLayer(channels, channels, kernel, 1)
        self.relu = nn.ReLU()
        self.conv2 = ConvLayer(channels, channels, kernel, 1)

    def forward(self, x):
        return self.conv2(self.relu(self.conv1(x))) + x


class DeconvLayer(nn.Module):
    """
    Convolución traspuesta (upsampling aprendido) + InstanceNorm. Es la
    operación inversa de una Conv con stride: duplica la resolución espacial.
    """
    def __init__(self, in_c, out_c, kernel, stride, output_padding,
                 norm='instance'):
        super().__init__()
        self.deconv = nn.ConvTranspose2d(in_c, out_c, kernel, stride,
                                         kernel // 2, output_padding)
        self.norm_type = norm
        if norm == 'instance':
            self.norm = nn.InstanceNorm2d(out_c, affine=True)
        elif norm == 'batch':
            self.norm = nn.BatchNorm2d(out_c, affine=True)

    def forward(self, x):
        x = self.deconv(x)
        return x if self.norm_type == 'None' else self.norm(x)

### Ejercicio 3 — Ensamblar la TransformerNet

**Objetivo:** Usar los bloques del setup (`ConvLayer`, `ResidualLayer`, `DeconvLayer`) para armar la red completa encoder → residuales → decoder.

**Enunciado:**

Completá la clase `TransformerNetwork(nn.Module)`. En `__init__` tenés que crear **tres** `nn.Sequential`:

1. `self.ConvBlock` — encoder:
   - `ConvLayer(3, 32, kernel=9, stride=1)` + `nn.ReLU()`
   - `ConvLayer(32, 64, kernel=3, stride=2)` + `nn.ReLU()`
   - `ConvLayer(64, 128, kernel=3, stride=2)` + `nn.ReLU()`
2. `self.ResidualBlock` — **5** `ResidualLayer(128, kernel=3)` apilados.
3. `self.DeconvBlock` — decoder:
   - `DeconvLayer(128, 64, kernel=3, stride=2, output_padding=1)` + `nn.ReLU()`
   - `DeconvLayer(64, 32, kernel=3, stride=2, output_padding=1)` + `nn.ReLU()`
   - `ConvLayer(32, 3, kernel=9, stride=1, norm='None')` — **sin** normalización ni ReLU: queremos que la salida sea una imagen, no features activadas.

En `forward` encadená los tres bloques en orden: `x → ConvBlock → ResidualBlock → DeconvBlock`.

> **Pista:** Para los 5 bloques residuales podés usar `nn.Sequential(*[ResidualLayer(128, 3) for _ in range(5)])`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

- ¿Cuál es el rol de cada uno de los tres bloques (encoder, residuales, decoder)? Pensá qué pasa con la resolución espacial y con el número de canales en cada uno.
- ¿Por qué el bloque del medio usa conexiones residuales y no convoluciones simples? ¿Qué le aporta al entrenamiento el hecho de que la identidad sea una solución trivial?

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Verificar la arquitectura

**Objetivo:** Instanciar la red, contar sus parámetros y verificar que un tensor de entrada sale con la misma forma que entró.

**Enunciado:**

1. Instanciá `TransformerNetwork()` y guardala en `transformerNet`.
2. Contá el total de parámetros entrenables (usá `sum(p.numel() for p in transformerNet.parameters())`) e imprimilo. Esperamos ~1.68 millones.
3. Generá un tensor de prueba `x = torch.randn(1, 3, 256, 256)`, pasalo por la red con `y = transformerNet(x)` e imprimí la forma de la salida. Tiene que ser **idéntica** a la de la entrada.

> **Pista:** Por ahora no movemos nada al GPU — este ejercicio es solo de verificación estructural.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

La red reduce la resolución espacial 4× en el encoder, opera en esa resolución reducida durante los bloques residuales, y la recupera en el decoder. ¿Por qué es importante que la **salida final tenga exactamente la misma forma que la entrada**? ¿Qué ventaja tiene hacer el trabajo "pesado" (bloques residuales) en la resolución reducida en lugar de a resolución completa?

*(Escribí tu respuesta acá)*

---
## Sección C: Entrenamiento

### El loop

Ya tenés todas las piezas:

- `gram`, `content_loss`, `style_loss` — las pérdidas perceptuales (Sección A).
- `TransformerNetwork` — la red que vamos a entrenar (Sección B).

Falta una última pieza preescrita: la **VGG-16** que calcula las features sobre las que medimos las pérdidas. Usamos los pesos Caffe-convertidos de Justin Johnson, que son los que esperan ser alimentados con imágenes BGR en rango [0, 255] menos la media de ImageNet (`[-103.939, -116.779, -123.68]` por canal). Ese preprocesamiento es el que asumimos en toda la sección.

In [ ]:
# ─── Setup: descarga y definición de la VGG-16 perceptual ──────────────────
# Los pesos de Justin Johnson son la versión Caffe-convertida de VGG-16. Son
# consistentes con el preprocesamiento BGR + mean-subtraction que usamos en
# el resto del laboratorio (itot/ttoi están en BGR por la lectura con cv2).
VGG_PATH = '/content/vgg16-00b39a1b.pth'
if not os.path.exists(VGG_PATH):
    !wget -q --show-progress https://web.eecs.umich.edu/~justincj/models/vgg16-00b39a1b.pth -O {VGG_PATH}


class VGG16(nn.Module):
    """
    VGG-16 congelada que devuelve, en un dict, las activaciones de relu1_2,
    relu2_2, relu3_3 y relu4_3 — las capas que el paper usa para medir
    contenido y estilo.
    """
    def __init__(self, vgg_path=VGG_PATH):
        super().__init__()
        vgg = models.vgg16(weights=None)
        vgg.load_state_dict(torch.load(vgg_path, weights_only=False),
                            strict=False)
        self.features = vgg.features
        for p in self.features.parameters():
            p.requires_grad = False  # red frozen: nunca se actualiza

    def forward(self, x):
        layers = {'3': 'relu1_2', '8': 'relu2_2',
                  '15': 'relu3_3', '22': 'relu4_3'}
        out = {}
        for name, layer in self.features._modules.items():
            x = layer(x)
            if name in layers:
                out[layers[name]] = x
                if name == '22':
                    break
        return out


print('VGG16 lista')

In [ ]:
# ─── Setup: hiperparámetros del entrenamiento ──────────────────────────────
# Los valores por defecto vienen del repo oficial del paper y funcionan bien
# sobre COCO val2017. Si querés experimentar, los pesos content/style son
# los más influyentes: subir STYLE_WEIGHT da un estilo más agresivo.
TRAIN_IMAGE_SIZE = 256
DATASET_PATH     = '/content/dataset'
STYLE_IMAGE_PATH = style_path

NUM_EPOCHS     = 1
BATCH_SIZE     = 4
CONTENT_WEIGHT = 50
STYLE_WEIGHT   = 50
ADAM_LR        = 1e-3

PRINT_EVERY    = 25
SAVE_MODEL_PATH = '/content/models/'
SAVE_IMAGE_PATH = '/content/samples/'
SEED = 35

os.makedirs(SAVE_MODEL_PATH, exist_ok=True)
os.makedirs(SAVE_IMAGE_PATH, exist_ok=True)

# Media de ImageNet (BGR) que se resta a las imágenes antes de pasarlas a VGG.
imagenet_neg_mean = torch.tensor(
    [-103.939, -116.779, -123.68], dtype=torch.float32
).reshape(1, 3, 1, 1).to(device)

# Semilla para reproducibilidad.
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed(SEED)
elif device == "mps":
    torch.mps.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

### Ejercicio 5 — Completar el loop de entrenamiento

**Objetivo:** Ensamblar todas las piezas en el loop de entrenamiento de Johnson y correrlo sobre COCO.

**Contexto:**

El loop hace, por cada batch de imágenes de contenido:

1. Pasar el batch por la **TransformerNet** → imagen generada.
2. Pasar contenido y generado por **VGG** → dos dicts de features.
3. Calcular `content_loss` entre los dos dicts.
4. Calcular `style_loss` entre el dict del generado y la Gram precomputada del estilo.
5. Combinar con los pesos `CONTENT_WEIGHT` y `STYLE_WEIGHT`, backward, step.

Las partes de infraestructura (DataLoader, precomputar la Gram del estilo, el chequeo de `cur_bs`, los prints periódicos, la conversión RGB→BGR con `[:, [2, 1, 0]]`) ya están resueltas. Tenés que completar el cuerpo del loop.

**Enunciado:**

Completá los `TODO` marcados en el código. Tres pasos:

1. **Forward de la transformer** sobre `content_batch`.
2. **Features con VGG** — pasar contenido y generado por `VGG` **después de restar la media de ImageNet** con `.add(imagenet_neg_mean)`. Guardar los dos dicts.
3. **Pérdidas** — usar `content_loss` y `style_loss` (del Ejercicio 2) y ponderar por `CONTENT_WEIGHT` y `STYLE_WEIGHT`.

Después corré la celda. Con `NUM_EPOCHS=1` y `BATCH_SIZE=4`, en T4 tarda ~5 minutos.

> **Pistas:**
> - La Gram del estilo se precalcula **antes** del loop y se indexa por los nombres de capa ya calculados.
> - `style_loss` espera recibir `style_gram` completo pero, como la imagen de estilo se expandió al tamaño del batch, necesitamos cortar a `cur_bs` (el batch actual puede ser menor en la última iteración): en el código ves `{k: v[:cur_bs] for k, v in style_gram.items()}`.

In [ ]:
def train():
    # ─── Dataset y dataloader ────────────────────────────────────────────────
    transform = transforms.Compose([
        transforms.Resize(TRAIN_IMAGE_SIZE),
        transforms.CenterCrop(TRAIN_IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.mul(255)),
    ])
    train_dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    total_iters = NUM_EPOCHS * len(train_loader)
    print(f'Dataset: {len(train_dataset)} imgs, '
          f'{len(train_loader)} iters/epoch, total={total_iters}')

    # ─── Modelos ────────────────────────────────────────────────────────────
    net = TransformerNetwork().to(device)
    VGG = VGG16().to(device)

    # ─── Gram de la imagen de estilo (precomputada) ─────────────────────────
    style_image = load_image(STYLE_IMAGE_PATH)
    style_tensor = itot(style_image).to(device).add(imagenet_neg_mean)
    B, C, H, W = style_tensor.shape
    style_features = VGG(style_tensor.expand([BATCH_SIZE, C, H, W]))
    style_gram = {k: gram(v) for k, v in style_features.items()}

    # ─── Optimizador ────────────────────────────────────────────────────────
    optimizer = optim.Adam(net.parameters(), lr=ADAM_LR)

    c_hist, s_hist, t_hist = [], [], []
    c_sum = s_sum = t_sum = 0.0
    batch_count = 1
    start = time.time()

    for epoch in range(NUM_EPOCHS):
        print(f'\n======== Epoch {epoch + 1}/{NUM_EPOCHS} ========')
        for content_batch, _ in train_loader:
            cur_bs = content_batch.shape[0]
            optimizer.zero_grad()

            # Las imágenes vienen en RGB; VGG Caffe espera BGR → intercambiamos canales.
            content_batch = content_batch[:, [2, 1, 0]].to(device)

            # ─── TODO (1): forward de la transformer ─────────────────────────
            generated_batch = None  # reemplazá esta línea

            # ─── TODO (2): features con VGG (restar la media antes de entrar) ─
            content_features = None    # reemplazá
            generated_features = None  # reemplazá

            # ─── TODO (3): pérdidas ──────────────────────────────────────────
            # Cortá style_gram a cur_bs para el último batch posiblemente más chico.
            sg_cur = {k: v[:cur_bs] for k, v in style_gram.items()}
            c_loss = None  # reemplazá  (usar content_loss * CONTENT_WEIGHT)
            s_loss = None  # reemplazá  (usar style_loss   * STYLE_WEIGHT)

            # ─── Backward y step (ya resuelto) ──────────────────────────────
            total_loss = c_loss + s_loss
            total_loss.backward()
            optimizer.step()

            c_sum += c_loss.item(); s_sum += s_loss.item()
            t_sum += total_loss.item()

            if batch_count % PRINT_EVERY == 0 or batch_count == total_iters:
                elapsed = time.time() - start
                eta = elapsed / batch_count * (total_iters - batch_count)
                print(f'[{batch_count:5d}/{total_iters}] '
                      f'content={c_sum / batch_count:.1f}  '
                      f'style={s_sum / batch_count:.1f}  '
                      f'total={t_sum / batch_count:.1f}  '
                      f'elapsed={elapsed:.0f}s  eta={eta:.0f}s')
                c_hist.append(c_sum / batch_count)
                s_hist.append(s_sum / batch_count)
                t_hist.append(t_sum / batch_count)

            batch_count += 1

    print(f'\nEntrenamiento terminado en {(time.time() - start) / 60:.1f} min')

    net.eval().cpu()
    final_path = f'{SAVE_MODEL_PATH}final.pth'
    torch.save(net.state_dict(), final_path)
    print(f'Pesos finales guardados: {final_path}')

    return c_hist, s_hist, t_hist


content_hist, style_hist, total_hist = train()

**Pregunta de análisis:**

Durante todo el entrenamiento, la **VGG está congelada** (sus pesos no cambian) y la **TransformerNet se entrena**. ¿Por qué usamos la VGG solo como **extractor de features** y no la entrenamos también? ¿Qué pasaría si la entrenáramos junto con la TransformerNet, usando la misma pérdida total?

*(Escribí tu respuesta acá)*

### Ejercicio 6 — Curvas de pérdida

**Objetivo:** Visualizar cómo evolucionan las tres pérdidas (contenido, estilo, total) a lo largo del entrenamiento.

**Enunciado:**

1. Armá una figura `plt.subplots(1, 3, figsize=(15, 4))`.
2. Ploteá `content_hist` en el primer eje, `style_hist` en el segundo, `total_hist` en el tercero.
3. A cada eje ponele título (`"Content loss"`, `"Style loss"`, `"Total loss"`), etiqueta `xlabel = f"checkpoint (cada {PRINT_EVERY} iters)"` y grilla.
4. `plt.tight_layout()` y `plt.show()`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Mirá las tres curvas con cuidado. Vas a ver que la **style loss baja** fuertemente, la **total loss baja** también, pero la **content loss sube** durante la primera mitad del entrenamiento antes de estabilizarse. ¿Qué está pasando? ¿Qué nos dice esto sobre el tradeoff intrínseco del problema que estamos resolviendo?

*(Escribí tu respuesta acá)*

---
## Sección D: Inferencia y análisis

### El pago: un solo forward pass

Ya entrenaste el modelo. La promesa del método de Johnson es que estilizar una imagen nueva cuesta **un solo forward pass** — sin optimización, sin gradientes, sin iteraciones. Lo vamos a comprobar ahora sobre 3 imágenes random del dataset.

### Ejercicio 7 — Estilizar imágenes nuevas

**Objetivo:** Cargar los pesos entrenados, escribir una función de inferencia, y aplicarla a 3 imágenes random de val2017 para visualizar los resultados.

**Enunciado:**

1. Cargá los pesos finales (`{SAVE_MODEL_PATH}final.pth`) en una nueva instancia de `TransformerNetwork` y mandala al `device`. Ponela en `.eval()`.
2. Definí la función `stylize(image_path, max_size=512)`:
   - Cargá la imagen con `load_image` (devuelve BGR).
   - Convertila a tensor con `itot(img, max_size=max_size)` y mandala al `device`.
   - Pasala por la red **dentro de `torch.no_grad()`** y recortá la salida a `[0, 255]` con `.clamp(0, 255)`.
   - Devolvé la tupla `(original_bgr, estilizada_numpy)` donde la segunda es un array numpy `H×W×3`.
3. Elegí 3 archivos random de `/content/dataset/val2017` y mostralos en una grilla `3 × 2` (original a la izquierda, estilizada a la derecha). Para mostrar tenés que convertir BGR → RGB con `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)` (la estilizada conviene castearla a `uint8` primero: `styled.clip(0, 255).astype('uint8')`).

> **Pista:** `torch.load(..., map_location=device, weights_only=True)` te carga los pesos en el dispositivo correcto en una línea.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

Mirando las 3 imágenes estilizadas, respondé:

- ¿Qué se **preserva** del contenido original (formas, posiciones, grandes estructuras)?
- ¿Qué se **transfiere** del estilo (colores, texturas, tipos de pincelada)?
- ¿Notás algún artefacto en las imágenes estilizadas? ¿A qué te parece que se debe?

*(Escribí tu respuesta acá)*

### Ejercicio 8 — Comparación Gatys vs. Johnson

**Objetivo:** Sintetizar las diferencias entre el método de la teoría (Gatys, optimización directa de píxeles) y el que acabás de implementar (Johnson, red feed-forward entrenada).

Este ejercicio es **solo de análisis**: respondé en la celda siguiente las cuatro preguntas de abajo.

1. **Tiempo**: ¿cuánto tarda Gatys en estilizar *una* imagen nueva? ¿Cuánto tarda Johnson? ¿Cuánto tardaron ustedes entrenando la TransformerNet recién?
2. **Flexibilidad**: si querés probar un nuevo estilo, ¿qué tenés que hacer con cada método?
3. **Calidad**: ¿en qué dimensiones podría ganar Gatys? ¿En qué dimensiones podría ganar Johnson?
4. **Cuándo usarías cada uno**: imaginá dos escenarios reales — (a) un artista que quiere experimentar con muchos estilos sobre la misma foto, (b) una app móvil que estiliza video en tiempo real. ¿Cuál método elegirías para cada uno y por qué?

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno con GPU activada y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] El entrenamiento corrió sobre `val2017` completo (1.250 iteraciones) y la curva de `total_loss` baja de forma estable.
- [ ] Las 3 imágenes estilizadas del Ejercicio 7 muestran claramente el estilo transferido.
- [ ] Los valores numéricos que imprimo son razonables (no hay infinitos ni `NaN`).
- [ ] Todos los gráficos tienen título, etiquetas en los ejes y grilla.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Entrenaste una red de transferencia de estilos desde cero. Implementaste las piezas que comparten los métodos de Gatys y Johnson (matriz de Gram y pérdidas perceptuales sobre features de VGG), ensamblaste una TransformerNet encoder-decoder con bloques residuales, y la entrenaste sobre COCO para reproducir un estilo específico en un solo forward pass.

En el próximo bloque vamos a ver modelos generativos más generales (autoencoders variacionales y GANs), donde la idea de "una red que genera imágenes" se profundiza hasta convertirse en el objetivo central, no solo un medio para transferir estilo.